# Hello World

In [ ]:
import torch

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

## Objective Function

Consider the toy problem

$$
\min{x \in \mathbb{R}^5} \, |x|_2.
$$

In [ ]:
class Model:
    def __init__(self):
        self.x = torch.randn(5, device=device)
        self.loss = None

    def evaluate(self):
        self.loss = torch.norm(self.x, p=2)
        n = self.x.norm()
        if n == 0:
            self.x.grad = torch.zeros_like(self.x)
        else:
            self.x.grad = self.x / n

        return self.loss

    def parameters(self):
        return [self.x]

model = Model()

## Optimiser

Let's use a standard quasi-newton line-search based optimiser with L-BFGS hessian approximation and Wolfe backtracking conditions for the line-search.

Since this toy problem is smooth and convex, we can the optimisation in a single step! Let's make sure that our step size is large enough so we initially step past the minimum, then rely on the backtracking to get us exactly at the minimum

In [ ]:
step_size = 100.0 # The line search will backtrack anyway!
opt = torch.optim.LBFGS(model.parameters(), lr=step_size, line_search_fn='strong_wolfe')

## Run the optimisation!

In [ ]:
def _closure():
    opt.zero_grad()
    return model.evaluate()

# Only need one iteration as the backtracking will take us to the minimum!
for _ in range(1):
    loss = opt.step(_closure)

print(model.x.detach())